# Langfuse Demo 1 — LLM Observability

**Dataset:** `ecommerce_support_requests.csv`  
**Runtime:** Python 3.11.9  
**Purpose:** A simple, instructor-led Langfuse demonstration.

## Architecture

```text
CSV support request → PII masking → prompt → OpenAI LLM → classification/response
                                              ↓
                         Langfuse trace: input, output, tokens, latency and cost
```

This notebook demonstrates direct LLM observability. The Langfuse OpenAI wrapper automatically captures the model call. We deliberately send only masked customer text to the trace.

## Step 1 — Install packages

Run once, then restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -q -U langfuse openai pandas python-dotenv

## Step 2 — Load the supplied e-commerce dataset

In [ ]:
import os, re, json
import pandas as pd

# Keep ecommerce_support_requests.csv in the same folder as this notebook.
CSV_FILE = "ecommerce_support_requests.csv"
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} rows from {CSV_FILE}")
display(df.head(3))

## Step 3 — Load credentials from `.env`

Create Langfuse keys in **Project Settings → API Keys** and store them in a `.env` file beside the notebook. The validation cell reports missing variable names without printing secret values.

In [ ]:
from dotenv import load_dotenv

# Loads variables from a .env file in the notebook's current directory.
load_dotenv()

required_keys = ["OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
missing_keys = [key for key in required_keys if not os.getenv(key)]
if missing_keys:
    raise ValueError(f"Missing variables in .env: {', '.join(missing_keys)}")

# Keep this in .env when using another Langfuse region or a self-hosted instance.
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

from langfuse import get_client
langfuse = get_client()
print("Langfuse authentication:", langfuse.auth_check())

## Step 4 — Mask PII before it enters prompts or traces

In [ ]:
def mask_pii(value):
    """Mask likely email addresses and 10-digit phone numbers before tracing."""
    text = str(value)
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "<EMAIL>", text)
    text = re.sub(r"(?<!\d)\d{10}(?!\d)", "<PHONE>", text)
    return text


sample = df.loc[2, "customer_message"]
print("Before:", sample)
print("After :", mask_pii(sample))

## Step 5 — Call the LLM through the Langfuse OpenAI wrapper

The `name`, `metadata` and `tags` fields make traces easy to filter in the dashboard.

In [ ]:
from langfuse.openai import OpenAI

client = OpenAI()
row = df.iloc[0]
safe_message = mask_pii(row["customer_message"])

response = client.chat.completions.create(
    name="ecommerce-support-llm",
    model="gpt-4.1-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": "Classify the issue and give a concise, helpful support reply. Do not invent order facts."},
        {"role": "user", "content": safe_message},
    ],
    metadata={"request_id": str(row["request_id"]), "issue_type": str(row["issue_type"])},
    tags=["training", "llm", "ecommerce"],
)
answer = response.choices[0].message.content
print(answer)
langfuse.flush()

## Step 6 — Run a small batch

Five separate traces let participants compare issue types, response time, tokens and cost.

In [ ]:
results = []
for _, row in df.head(5).iterrows():
    reply = client.chat.completions.create(
        name="ecommerce-support-batch",
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "Return JSON with keys category, priority, and reply."},
            {"role": "user", "content": mask_pii(row["customer_message"])},
        ],
        metadata={"request_id": str(row["request_id"])},
        tags=["training", "llm", "batch"],
    ).choices[0].message.content
    results.append({"request_id": row["request_id"], "model_output": reply})

langfuse.flush()
display(pd.DataFrame(results))

## Step 7 — Langfuse monitoring walkthrough

1. Open **Tracing / Traces** and select `ecommerce-support-llm`.
2. Check **Input and Output**: verify that the customer message is masked and the reply answers the request.
3. Check **Model**: confirm which OpenAI model handled the request.
4. Check **Latency**: identify slow responses and compare latency across the batch.
5. Check **Tokens**: compare input, output and total token consumption.
6. Check **Cost**: explain that Langfuse tracks the estimated OpenAI cost; Langfuse is not generating the response.
7. Check **Status and errors**: failed calls should show an error and help locate the failing step.
8. Check **Metadata**: inspect `request_id` and `issue_type` for troubleshooting.
9. Check **Tags**: filter using `training`, `llm` or `batch`.
10. Check **Scores**: add a manual `helpfulness` score or configure an LLM-as-a-judge evaluator.
11. Compare the five batch traces to spot unusually long, costly or poor responses.
12. Open **Dashboards** and graph request volume, latency, cost and quality score over time.

### Recommended LLM alerts

- High latency: response time exceeds the limit selected by your team.
- High cost: token or estimated cost exceeds the expected range.
- Generation error: the model call fails or returns no answer.
- Low quality: helpfulness, correctness or safety score falls below the accepted threshold.
- Unmasked PII: sensitive data appears in trace input or output.

**Teaching point:** Langfuse observes and evaluates the LLM application; it does not replace the LLM.